# Registro histórico: 11_oversampling_initial

Origem: `C53` — `Colab Notebooks/melanoma_classifier_efficientnet.ipynb`.

Tentativa de oversampling com separação por metadados.

**Limitações conhecidas:** Erro salvo de conjunto vazio; procura imagens na raiz, enquanto a estrutura baixada usa subpastas.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
import os
import random
import shutil
import zipfile
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import requests
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
from torchvision import transforms, datasets
from torchvision.models import EfficientNet_B0_Weights # Alterado para EfficientNet_B0_Weights
from torch.optim.lr_scheduler import ReduceLROnPlateau

## Configurações Iniciais

In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Set random seeds for reproducibility
SEED = 91432
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

## Download e Preparação do Dataset HAM10000 do Kaggle

In [ ]:
# Célula para baixar o dataset HAM10000 do Kaggle

from pathlib import Path
import os

# Instalar kaggle (se necessário)
!pip install -q kaggle

# Fazer upload do kaggle.json
# Vá em: Kaggle → Settings → API → Create New API Token
from google.colab import files
print("📤 Faça upload do seu arquivo kaggle.json:")
uploaded = files.upload()

# Configurar credenciais do Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Baixar e extrair o dataset
print("\n📦 Baixando dataset HAM10000 do Kaggle...")
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

print("\n📂 Extraindo arquivos...")
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/HAM10000

# Definir caminhos
BASE_DIR = Path('/content/HAM10000')
METADATA_FILE = BASE_DIR / 'HAM10000_metadata.csv'
IMAGE_DIR = BASE_DIR  # As imagens ficam na raiz do dataset

# Verificar estrutura
print(f"\n✅ Dataset pronto!")
print(f"📄 Metadados: {METADATA_FILE}")
print(f"📂 Imagens em: {IMAGE_DIR}")
print(f"\n📋 Arquivos disponíveis:")
!ls -lh /content/HAM10000/*.csv

## Processamento de Metadados e Divisão de Dados

In [ ]:
try:
    df = pd.read_csv(METADATA_FILE)
except FileNotFoundError:
    print("ERRO: Arquivo de metadados não encontrado. Certifique-se de que o dataset foi baixado e extraído corretamente.")
    exit()

# Mapeamento para classificação binária: Melanoma (MEL) vs. Não Melanoma (NON-MEL)
# Classes do HAM10000: 'nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df'
# Melanoma (MEL) = 'mel'
# Não Melanoma (NON-MEL) = todas as outras classes
df['binary_label'] = df['dx'].apply(lambda x: 'melanoma' if x == 'mel' else 'non_melanoma')

# Contagem de classes binárias
print("\nContagem de classes binárias antes do oversampling:")
print(df['binary_label'].value_counts())

# Criar um DataFrame com o caminho completo das imagens
# O nome do arquivo de imagem é 'image_id.jpg'
df['path'] = df['image_id'].apply(lambda x: str(IMAGE_DIR / f"{x}.jpg"))

# Remover linhas onde o arquivo de imagem não foi encontrado (se houver)
df = df[df['path'].apply(lambda x: os.path.exists(x))]

# Divisão em treino, validação e teste (80/10/10)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['binary_label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['binary_label'])

print(f"\nTamanho do conjunto de Treino: {len(train_df)}")
print(f"Tamanho do conjunto de Validação: {len(val_df)}")
print(f"Tamanho do conjunto de Teste: {len(test_df)}")

## Oversampling (Duplicação da Classe Minoritária)

In [ ]:
class OversampledImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
        self.class_to_idx = {'non_melanoma': 0, 'melanoma': 1}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

        # Identificar a classe minoritária e majoritária
        counts = self.dataframe['binary_label'].value_counts()
        self.minority_class = counts.idxmin()
        self.majority_class = counts.idxmax()
        self.majority_count = counts.max()
        self.minority_count = counts.min()

        # Filtrar a classe minoritária
        self.minority_df = self.dataframe[self.dataframe['binary_label'] == self.minority_class]

        # Calcular o fator de duplicação
        self.duplication_factor = self.majority_count // self.minority_count
        self.remainder = self.majority_count % self.minority_count

        print(f"Oversampling: {self.minority_class} ({self.minority_count}) será duplicada {self.duplication_factor} vezes + {self.remainder} amostras.")

        # Criar a lista final de índices (incluindo as duplicações)
        self.indices = list(self.dataframe.index)
        minority_indices = list(self.minority_df.index)

        # Adicionar as duplicações
        for _ in range(self.duplication_factor): # Duplica o fator inteiro
            self.indices.extend(minority_indices)

        # Adicionar o restante das amostras
        self.indices.extend(random.sample(minority_indices, self.remainder))

        # O conjunto final de treino será composto por:
        # - Todas as amostras originais
        # - As duplicações da classe minoritária
        self.final_df = self.dataframe.loc[self.indices].reset_index(drop=True)
        print(f"Tamanho final do dataset de treino após oversampling: {len(self.final_df)}")
        print("Nova contagem de classes no treino:")
        print(self.final_df['binary_label'].value_counts())


    def __len__(self):
        return len(self.final_df)

    def __getitem__(self, idx):
        row = self.final_df.iloc[idx]
        img_path = row['path']
        label_name = row['binary_label']
        label = self.class_to_idx[label_name]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

# Corrigir o val/test dataset para não fazer oversampling (apenas para usar a mesma classe)
class SimpleImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
        self.class_to_idx = {'non_melanoma': 0, 'melanoma': 1}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = row['path']
        label_name = row['binary_label']
        label = self.class_to_idx[label_name]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

## Data Transforms e DataLoaders

In [ ]:
# EfficientNet-B0 usa 224x224 por padrão.
# Usaremos os transforms recomendados pelos pesos pré-treinados para normalização.
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
auto_transforms = weights.transforms()

# Modificando os transforms para incluir mais data augmentation
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    auto_transforms.normalize # Usar a normalização padrão do EfficientNet
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    auto_transforms.normalize
])

# Criar Datasets
train_dataset = OversampledImageDataset(train_df, transform=train_transform)
val_dataset = SimpleImageDataset(val_df, transform=test_transform)
test_dataset = SimpleImageDataset(test_df, transform=test_transform)

BATCH_SIZE = 32

# Criar DataLoaders
train_dataloader = DataLoader(dataset=train_dataset,
                             batch_size=BATCH_SIZE,
                             shuffle=True,
                             num_workers=os.cpu_count() // 2 or 1,
                             pin_memory=True if device == "cuda" else False)

val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=BATCH_SIZE,
                            shuffle=False,
                            num_workers=os.cpu_count() // 2 or 1,
                            pin_memory=True if device == "cuda" else False)

test_dataloader = DataLoader(dataset=test_dataset,
                           batch_size=BATCH_SIZE,
                           shuffle=False,
                           num_workers=os.cpu_count() // 2 or 1,
                           pin_memory=True if device == "cuda" else False)

print(f"DataLoaders criados - Treino: {len(train_dataloader)} batches, Validação: {len(val_dataloader)} batches, Teste: {len(test_dataloader)} batches")

## Criação do Modelo EfficientNet-B0

In [ ]:
def create_efficientnet_b0_model(num_classes=2, pretrained=True):
    """Cria e modifica EfficientNet-B0 para classificação binária."""
    if pretrained:
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        print("EfficientNet-B0 carregado com pesos pré-treinados do ImageNet.")
    else:
        model = models.efficientnet_b0(weights=None)
        print("EfficientNet-B0 criado sem pesos pré-treinados.")

    # Congelar camadas iniciais para transfer learning
    for param in model.features.parameters():
        param.requires_grad = False

    # Modificar o classificador para 2 classes
    # O classificador do EfficientNet é uma sequência: (Dropout, Linear)
    num_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(num_features, num_classes)

    print(f"Classificador modificado para {num_classes} classes.")
    return model

model = create_efficientnet_b0_model(num_classes=1, pretrained=True) # 1 classe de saída para BCEWithLogitsLoss
model = model.to(device)

## Setup do Treinamento Otimizado

In [ ]:
# Função de Perda: BCEWithLogitsLoss é ideal para classificação binária e permite pesos
# para lidar com o desbalanceamento.
class_counts = train_df['binary_label'].value_counts()
total_samples = class_counts.sum()
# Peso = Contagem da Classe Majoritária / Contagem da Classe Minoritária
# A classe positiva (Melanoma) é a minoritária.
pos_weight_value = class_counts['non_melanoma'] / class_counts['melanoma']

pos_weight = torch.tensor([pos_weight_value], dtype=torch.float).to(device)
print(f"Peso da classe positiva (Melanoma) para BCEWithLogitsLoss: {pos_weight.item():.2f}")

# Usamos BCEWithLogitsLoss, que combina Sigmoid e Binary Cross Entropy
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Otimizador: AdamW com baixo Learning Rate para fine-tuning
optimizer = optim.AdamW(model.parameters(),
                       lr=1e-4,
                       weight_decay=1e-4)

# Scheduler: Reduz o LR quando a métrica de validação estagnar
scheduler = ReduceLROnPlateau(optimizer,
                              mode='max', # Monitorar o F1-Score (maximização)
                              factor=0.5,
                              patience=3,
                              verbose=True)

## Funções de Métrica e Callback de F1-Score

In [ ]:
def calculate_metrics(y_true, y_pred_logits):
    """Calcula F1-Score, Acurácia e Loss para um batch."""
    # Para BCEWithLogitsLoss, precisamos aplicar Sigmoid e arredondar para obter as previsões binárias
    y_pred_probs = torch.sigmoid(y_pred_logits)
    y_pred_labels = torch.round(y_pred_probs).squeeze()

    # Converter para numpy para usar f1_score do sklearn
    y_true_np = y_true.cpu().numpy()
    y_pred_labels_np = y_pred_labels.cpu().numpy()

    # F1-Score (macro para lidar com o desbalanceamento)
    f1 = f1_score(y_true_np, y_pred_labels_np, average='macro', zero_division=0)
    acc = (y_pred_labels == y_true).float().mean()

    return f1, acc

def val_step(model: torch.nn.Module,
             data_loader: torch.utils.data.DataLoader,
             loss_fn: torch.nn.Module,
             device: torch.device = device):
    """Passo de validação."""
    model.eval()
    val_loss, val_f1, val_acc = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.inference_mode():
        for X, y in data_loader:
            # y deve ser float e ter dimensão [batch_size, 1] para BCEWithLogitsLoss
            X, y = X.to(device), y.to(device).float().unsqueeze(1)

            # 1. Forward pass
            y_logits = model(X)
            loss = loss_fn(y_logits, y)
            val_loss += loss.item()

            # 2. Calcular métricas
            f1, acc = calculate_metrics(y, y_logits)
            val_f1 += f1
            val_acc += acc.item()

            # Coletar previsões e labels para o relatório final
            all_labels.extend(y.cpu().squeeze().tolist())
            all_preds.extend(torch.round(torch.sigmoid(y_logits)).cpu().squeeze().tolist())

    # Ajustar métricas para o tamanho do dataloader
    val_loss /= len(data_loader)
    val_f1 /= len(data_loader)
    val_acc /= len(data_loader)

    return val_loss, val_f1, val_acc, all_labels, all_preds

def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device = device):
    """Passo de treinamento."""
    model.train()
    train_loss, train_f1, train_acc = 0, 0, 0

    progress_bar = tqdm(data_loader, desc="Training", leave=False)

    for X, y in progress_bar:
        X, y = X.to(device), y.to(device).float().unsqueeze(1)

        # 1. Forward pass
        y_logits = model(X)
        loss = loss_fn(y_logits, y)
        train_loss += loss.item()

        # 2. Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 3. Calcular métricas
        f1, acc = calculate_metrics(y, y_logits)
        train_f1 += f1
        train_acc += acc.item()

        progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'f1': f'{f1:.4f}'})

    # Ajustar métricas para o tamanho do dataloader
    train_loss /= len(data_loader)
    train_f1 /= len(data_loader)
    train_acc /= len(data_loader)

    return train_loss, train_f1, train_acc

## Early Stopping Customizado (F1-Score)

In [ ]:
class F1EarlyStopping:
    """Early stops the training if validation F1-Score does not improve after a given patience."""
    def __init__(self, patience=5, min_f1=0.75, verbose=False, delta=0):
        self.patience = patience
        self.min_f1 = min_f1
        self.verbose = verbose
        self.counter = 0
        self.best_f1 = -np.Inf
        self.early_stop = False
        self.delta = delta
        self.best_model_state = None

    def __call__(self, val_f1, model):
        if val_f1 > self.best_f1 + self.delta:
            self.best_f1 = val_f1
            self.counter = 0
            self.best_model_state = model.state_dict()
            if self.verbose:
                print(f'Validation F1-Score improved ({self.best_f1:.4f}). Saving model state.')
        else:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

        # Condição adicional: Parar se o F1-Score mínimo for atingido
        if val_f1 >= self.min_f1:
            print(f"Target F1-Score of {self.min_f1} reached! Stopping training.")
            self.early_stop = True

        return self.early_stop

## Loop de Treinamento Principal

In [ ]:
EPOCHS = 25 # Definir um número máximo de épocas
early_stopping = F1EarlyStopping(patience=5, min_f1=0.75, verbose=True)

train_losses, val_losses = [], []
train_f1s, val_f1s = [], []
train_accs, val_accs = [], []

start_time = timer()

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")

    # Treinamento
    train_loss, train_f1, train_acc = train_step(model, train_dataloader, loss_fn, optimizer, device)
    train_losses.append(train_loss)
    train_f1s.append(train_f1)
    train_accs.append(train_acc)

    # Validação
    val_loss, val_f1, val_acc, _, _ = val_step(model, val_dataloader, loss_fn, device)
    val_losses.append(val_loss)
    val_f1s.append(val_f1)
    val_accs.append(val_acc)

    # Ajustar Learning Rate com base no F1-Score de validação
    scheduler.step(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc*100:.2f}%")

    # Early Stopping
    if early_stopping(val_f1, model):
        print("Early stopping at epoch", epoch + 1)
        # Carregar o melhor modelo
        if early_stopping.best_model_state is not None:
            model.load_state_dict(early_stopping.best_model_state)
        break

end_time = timer()
total_train_time = end_time - start_time
print(f"\nTempo total de treinamento: {total_train_time:.2f} segundos")

## Avaliação no Conjunto de Teste

In [ ]:
print("\nAvaliação no conjunto de Teste...")
test_loss, test_f1, test_acc, test_labels, test_preds = val_step(model, test_dataloader, loss_fn, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1-Score (Macro): {test_f1:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}%")

## Relatório de Classificação e Matriz de Confusão

In [ ]:
class_names = ['non_melanoma', 'melanoma']
print("\n📈 Detailed Classification Report:")
print(classification_report(test_labels, test_preds, target_names=class_names, zero_division=0))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig("confusion_matrix_efficientnet.png")
# plt.show() # Comentado para evitar erro em ambiente sem display

## Salvamento do Modelo e Histórico

In [ ]:
MODEL_SAVE_PATH = Path("models")
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)
MODEL_NAME = f"EfficientNetB0_Melanoma_F1_{test_f1:.4f}_E{epoch+1}.pth"
MODEL_FULL_PATH = MODEL_SAVE_PATH / MODEL_NAME

torch.save(obj=model.state_dict(), f=MODEL_FULL_PATH)
print(f"\nModelo salvo em: {MODEL_FULL_PATH}")

# Salvar histórico
history = {
    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_f1s": train_f1s,
    "val_f1s": val_f1s,
    "train_accs": train_accs,
    "val_accs": val_accs,
    "test_results": {
        "loss": test_loss,
        "f1_score": test_f1,
        "accuracy": test_acc
    },
    "training_time": total_train_time,
    "epochs_trained": epoch + 1,
    "best_val_f1": early_stopping.best_f1
}

import json
with open(MODEL_SAVE_PATH / "training_history_efficientnet.json", "w") as f:
    json.dump(history, f, indent=2)

print("Histórico de treinamento salvo.")

## Plotagem do Histórico

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_f1s, label='Train F1')
plt.plot(val_f1s, label='Validation F1')
plt.title('F1-Score vs. Epochs')
plt.xlabel('Epochs')
plt.ylabel('F1-Score (Macro)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss vs. Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss (BCEWithLogitsLoss)')
plt.legend()

plt.tight_layout()
plt.savefig("training_history_efficientnet.png")
# plt.show() # Comentado para evitar erro em ambiente sem display

print("\nScript concluído. O modelo e os gráficos foram salvos.")